In [ ]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report
from xgboost import XGBClassifier
import joblib
import time

df = pd.read_csv('../data/CIC-IDS-17_CLEANED.csv')
print(df.shape)
print(df['Label'].value_counts())

(925741, 71)
Label
BENIGN                      500000
DoS Hulk                    172846
DDoS                        128014
PortScan                     90694
DoS GoldenEye                10286
FTP-Patator                   5931
DoS slowloris                 5385
DoS Slowhttptest              5228
SSH-Patator                   3219
Bot                           1948
Web Attack - Brute Force      1470
Web Attack - XSS               652
Other Attack                    68
Name: count, dtype: int64


In [3]:
X = df.drop(columns = ['Label'])
Y = df['Label']

le = LabelEncoder()
y_encoded = le.fit_transform(Y)

joblib.dump(le, '../models/label_encoder.pkl')
print("Classes: ", le.classes_)

Classes:  ['BENIGN' 'Bot' 'DDoS' 'DoS GoldenEye' 'DoS Hulk' 'DoS Slowhttptest'
 'DoS slowloris' 'FTP-Patator' 'Other Attack' 'PortScan' 'SSH-Patator'
 'Web Attack - Brute Force' 'Web Attack - XSS']


In [4]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X, y_encoded,
    test_size=0.2,
    random_state=42,
    stratify = y_encoded
)

print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")

Training samples: 740592
Testing samples: 185149


In [5]:
models = {
    'Random Forest': RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1,
        class_weight='balanced'
    ),

    'XGBoost': XGBClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1,
        eval_metric='mlogloss',
        max_depth=6,
        learning_rate=0.1,
    ),

    'Decision Tree': DecisionTreeClassifier(
        random_state=42,
        class_weight='balanced',
    )
}

results = {}

for name, model in models.items():
    print(f"\n{'='*60}")
    print(f"  {name}")
    print(f"{'='*60}")

    start_time = time.time()
    model.fit(X_train, Y_train)
    training_time = time.time() - start_time

    start_time = time.time()
    y_pred = model.predict(X_test)
    predict_time = time.time() - start_time

    results[name] = {
        'model': model,
        'training_time': training_time,
        'predict_time': predict_time,
        'report': classification_report(Y_test, y_pred, target_names=le.classes_)
    }

    print(f"Train time: {training_time:.2f}s")
    print(f"Predict time: {predict_time:.2f}s")
    print(f"\n--- Classification Report ---\n")
    print(results[name]['report'])

print(f"\n{'='*60}")
print("  SUMMARY")
print(f"{'='*60}")
for name, result in results.items():
    print(f"  {name:<20} Train: {result['training_time']:.2f}s    Predict: {result['predict_time']:.2f}s")
print(f"{'='*60}\n")



  Random Forest
Train time: 18.27s
Predict time: 0.26s

--- Classification Report ---

                          precision    recall  f1-score   support

                  BENIGN       1.00      1.00      1.00    100000
                     Bot       0.88      0.91      0.89       390
                    DDoS       1.00      1.00      1.00     25603
           DoS GoldenEye       1.00      0.99      0.99      2057
                DoS Hulk       1.00      1.00      1.00     34569
        DoS Slowhttptest       0.99      0.99      0.99      1046
           DoS slowloris       1.00      0.99      0.99      1077
             FTP-Patator       1.00      1.00      1.00      1186
            Other Attack       1.00      0.50      0.67        14
                PortScan       1.00      1.00      1.00     18139
             SSH-Patator       1.00      1.00      1.00       644
Web Attack - Brute Force       0.74      0.87      0.80       294
        Web Attack - XSS       0.49      0.27      0.

In [6]:
best_model = results['Random Forest']['model']
joblib.dump(best_model, '../models/nids.pkl')

print("Model saved.")

Model saved.


In [7]:
#Find exact feature order for use in feature extraction
print(list(X_train.columns))

['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Fwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count', 'CWE Flag Count', 'ECE Flag Count', 'Down/Up Ratio', 'Average Packe